In [25]:
from langchain.agents import tool,create_react_agent,AgentExecutor
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain import hub
from langchain_core.messages import SystemMessage
from langchain_core.prompts.chat import ChatPromptTemplate,MessagesPlaceholder
from pprint import pprint
import requests
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
TMDB_API_KEY = os.getenv("TMDB_API_KEY")

In [26]:
def get_movie_recommendations(genre="action", year=None):
    """Fetches movie recommendations from TMDB."""
    genre_map = {
        "action": 28,
        "comedy": 35,
        "drama": 18,
        "thriller": 53,
        "horror": 27,
        "sci-fi": 878,
        "animation": 16
    }
    
    genre_id = genre_map.get(genre.lower(), None)
    if genre_id is None:
        return f"Genre '{genre}' not found in list."

    url = "https://api.themoviedb.org/3/discover/movie"
    params = {
        "api_key": TMDB_API_KEY,
        "sort_by": "popularity.desc",
        "with_genres": genre_id,
        "include_adult": "false",
        "language": "en-US",
        "page": 1
    }
    
    if year:
        params["primary_release_year"] = year

    response = requests.get(url, params=params)
    data = response.json()
    
    results = data.get("results", [])
    if not results:
        return f"No {genre} movies found for year {year}."

    return [f"{movie['title']} ({movie.get('release_date', 'N/A')})" for movie in results[:5]]


In [27]:
# tool
@tool
def recommend_movies(input_str: str):
    """Recommend top movies based on a genre and optional year."""
    parts = [p.strip() for p in input_str.split(",")]
    genre = parts[0]
    year = None
    if len(parts) > 1:
        # Extract only digits from the year part
        import re
        match = re.search(r"\d{4}", parts[1])
        if match:
            year = int(match.group())
    return get_movie_recommendations(genre, year)


In [28]:
# tool 
# api_wrapper = WikipediaAPIWrapper()

# wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

# @tool("Wikipedia Search")
# def search_wiki(query):
#     """ Searches Wikipedia based on the query """
#     return wiki_tool.invoke(query)

# chat_prompt = ChatPromptTemplate.from_messages([
#     ("system", "You are a helpful assistant. Think step-by-step."),
#     ("human", "{input}"),
#     ("placeholder", "{agent_scratchpad}")
# ])

# # Wrap prompt in a SystemMessage
# system_msg = SystemMessage(content=chat_prompt)
tools = [recommend_movies]

from langchain_core.prompts import PromptTemplate

template = '''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}'''

prompt = PromptTemplate.from_template(template)


# initialize llm
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite",
#     temperature=0.7,
#     google_api_key=api_key,
# )

llm = ChatGroq(
    model = "gemma2-9b-it",
    temperature = 0.7,
    api_key = groq_api_key
)

In [29]:
# initialize an agent

agent = create_react_agent(
    llm,
    tools,
    prompt
)

agent_executor = AgentExecutor(agent = agent,tools = tools)

In [30]:
response = agent_executor.invoke({"input": "Recommend me 5 drama movies"})
pprint(response)

{'input': 'Recommend me 5 drama movies',
 'output': "['F1 (2025-06-25)', 'Tuhog (2023-11-03)', 'Gold Rush Gang "
           '(2025-08-19)\', \'The Thursday Murder Club (2025-08-22)\', "God\'s '
           'Not Dead: In God We Trust (2024-09-12)"]'}
